# Results paragraph: Cross-metric correspondence of age effects

Uses **pooled (harmonized, non-vendorwise) age effects** per bundle and metric. Cross-metric correspondence = Spearman ρ between each metric's vector of bundle-wise age effects (Fig. 5d). Data: `data/age_quality_effects/age_quality_effects_all_outputs.rds`. Filter: `qc_metric == "no_quality"`, `source == "harmonized"`, `output_type == "non_vendorwise_pairwise"`, `scanner_manufacturer == "all"`, five metrics.

**Fig. 5:** Panel D = heatmap of cross-metric ρ; Panel E = best pair (RTOP–MKT); Panel F = worst pair (FA–MD).

## Setup and load pooled age effects

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

age_rds <- fs::path(project_root, "data", "age_quality_effects", "age_quality_effects_all_outputs.rds")
if (!file.exists(age_rds)) stop("Age effects file not found: ", age_rds)

df_all <- readRDS(age_rds)
if (!is.data.frame(df_all)) stop("Age effects file is not a data.frame.")

qc_target <- "no_quality"
source_target <- "harmonized"
output_target <- "non_vendorwise_pairwise"
scanner_target <- "all"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT",
  NODDI_icvf = "ICVF",
  MAPMRI_rtop = "RTOP",
  GQI_fa = "FA",
  GQI_md = "MD"
)
metric_order <- c("MKT", "ICVF", "RTOP", "FA", "MD")

qc_col <- if ("qc_metric" %in% names(df_all)) "qc_metric" else "qc_covariate"
if (!qc_col %in% names(df_all) || !qc_target %in% df_all[[qc_col]]) stop("No no_quality rows in age effects.")

df <- df_all %>%
  filter(
    .data[[qc_col]] == qc_target,
    .data[["source"]] == source_target,
    .data[["output_type"]] == output_target,
    .data[["scanner_manufacturer"]] == scanner_target,
    metric %in% metrics_keep,
    !is.na(bundle),
    !is.na(.data[["age_effect_size"]])
  ) %>%
  transmute(
    bundle = bundle,
    metric_label = unname(metric_labels[metric]),
    age_effect_size = as.numeric(.data[["age_effect_size"]])
  )

wide_effects <- df %>%
  distinct(bundle, metric_label, age_effect_size) %>%
  tidyr::pivot_wider(names_from = metric_label, values_from = age_effect_size) %>%
  tidyr::drop_na(all_of(metric_order))

if (nrow(wide_effects) < 10) stop("Too few complete bundles for cross-metric correlations.")
cat("N bundles (complete on all five metrics):", nrow(wide_effects), "\n")

## Cross-metric Spearman ρ and paragraph numbers

Compute Spearman ρ for all metric pairs. Extract: (1) minimum ρ among advanced metrics (ICVF, RTOP, MKT) for "ρ > X.X"; (2) average ρ of MD with other measures; (3) strongest pair (RTOP–MKT) and weakest pair (FA–MD) for Fig. 5e/f.

In [ ]:
pair_rho <- tidyr::expand_grid(metric_x = metric_order, metric_y = metric_order) %>%
  filter(metric_x != metric_y) %>%
  mutate(
    rho = purrr::map2_dbl(metric_x, metric_y, function(mx, my) {
      d <- wide_effects %>% select(all_of(c(mx, my))) %>% tidyr::drop_na()
      if (nrow(d) < 3) return(NA_real_)
      cor(d[[mx]], d[[my]], method = "spearman")
    })
  ) %>%
  filter(metric_x < metric_y) %>%
  mutate(pair = paste(metric_x, metric_y, sep = "-"))

advanced_metrics <- c("ICVF", "RTOP", "MKT")
advanced_pairs <- pair_rho %>%
  filter(
    (metric_x %in% advanced_metrics & metric_y %in% advanced_metrics)
  )
min_advanced_rho <- min(advanced_pairs$rho, na.rm = TRUE)
advanced_threshold <- round(min_advanced_rho, 2)

md_pairs <- pair_rho %>% filter(metric_x == "MD" | metric_y == "MD")
md_avg_rho <- mean(md_pairs$rho, na.rm = TRUE)
md_avg_rho_fmt <- round(md_avg_rho, 2)

best_pair_row <- pair_rho %>% filter(!is.na(rho)) %>% arrange(desc(rho)) %>% slice(1)
worst_pair_row <- pair_rho %>% filter(!is.na(rho)) %>% arrange(rho) %>% slice(1)

rho_rtop_mkt <- pair_rho %>%
  filter(
    (metric_x == "RTOP" & metric_y == "MKT") | (metric_x == "MKT" & metric_y == "RTOP")
  ) %>%
  pull(rho) %>%
  `[`(1)
rho_fa_md <- pair_rho %>%
  filter(
    (metric_x == "FA" & metric_y == "MD") | (metric_x == "MD" & metric_y == "FA")
  ) %>%
  pull(rho) %>%
  `[`(1)

rho_rtop_mkt_fmt <- round(rho_rtop_mkt, 2)
rho_fa_md_fmt <- round(rho_fa_md, 2)

cat("Advanced (ICVF, RTOP, MKT) minimum pairwise rho:", advanced_threshold, "=> paragraph 'rho >", advanced_threshold, "'\n")
cat("MD average rho with other metrics:", md_avg_rho_fmt, "\n")
cat("RTOP-MKT rho (strongest):", rho_rtop_mkt_fmt, "\n")
cat("FA-MD rho (weakest):", rho_fa_md_fmt, "\n")
cat("\nAll pairwise Spearman rho (upper triangle):\n")
print(pair_rho %>% select(metric_x, metric_y, rho) %>% mutate(rho = round(rho, 2)))

## Filled paragraph

In [ ]:
txt <- paste(
  "Variation in scanner vendors and the use of disparate microstructural modeling frameworks present major obstacles to synthesizing findings across the diffusion MRI literature, extending beyond multisite studies alone. We have thus far demonstrated that scanner-related variance can be mitigated through harmonization, that harmonization enhances the reproducibility of developmental effects across vendors, and that advanced microstructural measures exhibit greater sensitivity to age than tensor-derived metrics. However, it remains unclear whether the spatial patterning of developmental effects is consistent across modeling frameworks, or rather whether tensor-derived age effects are simply attenuated versions of a shared underlying gradient. To quantify cross-metric correspondence, we calculated the Spearman rank correlation between each metric's vector of bundle-wise age effects estimated from the full harmonized dataset (Fig. 5d). The advanced microstructural measures (ICVF, RTOP, and MKT) were highly correlated with one another (ρ >", advanced_threshold, "), indicating strong agreement in their spatial distribution of developmental effects. In contrast, tensor-derived metrics were more variable, with MD showing particularly weak correspondence with other measures (average ρ =", md_avg_rho_fmt, "). The strongest association was observed between RTOP and MKT (ρ =", rho_rtop_mkt_fmt, "; Fig. 5e), whereas the weakest correspondence was observed between FA and MD (ρ =", rho_fa_md_fmt, "; Fig. 5f). These findings indicate that the spatial organization of developmental effects is more consistent across advanced diffusion models than across tensor-derived measures.",
  sep = ""
)
cat(txt, "\n")